[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/pytorch/lab-p1-storage-and-strides.ipynb)

# LAB·P1 · Storage and strides, by hand

**Hardware:** any machine. Every cell here runs on plain CPU tensors; nothing needs a TPU or even a GPU.

A PyTorch tensor is three things bolted together: a storage (the actual bytes), a shape, and a set of strides (the map from an index to a byte offset). Most of what feels strange about views, `expand`, and in-place ops falls out of that one fact. This lab has you predict the numbers before torch prints them, then checks your read of the map against the real thing.

Before every reveal cell there is an empty "your prediction" cell above it. Write your answer there, then run the reveal and compare.

In [ ]:
import torch

print(torch.__version__)

## A view is a new map over the same bytes

`x.t()` does not move a single number. It returns a tensor that shares `x`'s storage and carries a different stride tuple, so reading it in the new order is a map problem, not a copy problem. `is_contiguous()` asks whether that map still matches a plain row-major layout for the current shape.

**your prediction:**

For `x = torch.arange(12.).reshape(3, 4)` and `v = x.t()`, write down `v.stride()` and whether `v.is_contiguous()` reads True or False. Then run the next cell.

In [ ]:
x = torch.arange(12.).reshape(3, 4)
v = x.t()                    # a view: same storage, remapped
print(v.stride(), v.is_contiguous())   # (1, 4) False

`v.stride()` comes out `(1, 4)`: step one element to move down a column of `v`, step four to move across a row, because that is exactly how `x`'s rows sit in memory. The map changed; the bytes did not. `is_contiguous()` reads False because no plain row-major stride tuple matches this shape over this storage.

`.contiguous()` is the tensor that fixes the mismatch: a copy laid out row-major for the shape you asked for.

**your prediction:**

Write down `c.stride()` for `c = v.contiguous()`, and say whether `c` and `v` share a storage.

In [ ]:
c = v.contiguous()           # a copy that owns a row-major layout
print(c.stride())            # (4, 1)

## Proving storage identity

Stride tuples are convincing, but indirect. `data_ptr()` reports the memory address of a tensor's first element, so two tensors that share a storage and an offset report the same address. A copy reports a different one.

**your prediction:**

Compare `x.data_ptr()` against `v.data_ptr()`, then compare `x.data_ptr()` against `c.data_ptr()`. Equal or not, for each pair?

In [ ]:
print(x.data_ptr() == v.data_ptr())   # True: v is a view over x's storage
print(x.data_ptr() == c.data_ptr())   # False: contiguous() copied

## expand vs reshape

`expand` only ever sets a stride of zero on a broadcast dimension, so it is free no matter what the input layout is: never a copy. `reshape` asks for a specific new shape, and it returns a view when the existing strides can express that shape; when they cannot, it copies, silently.

**your prediction:**

`row = x[:, :1].expand(3, 4)` reads four columns out of a tensor that only has one. Does this copy? Then: does `v.reshape(-1)`, flattening the non-contiguous view `v` down to one dimension, share `v`'s storage, or does it copy? Write both answers before running.

In [ ]:
row = x[:, :1].expand(3, 4)
print(row.data_ptr() == x.data_ptr())   # True: expand never copies

flat = v.reshape(-1)
print(flat.data_ptr() == v.data_ptr())   # False: v's strides can't express a flat view, so reshape copies

`expand` stayed free because it only sets a stride of zero on the broadcast dimension: still the same bytes, read more than once. `reshape` on `v` had no legal stride tuple that turns a transposed layout into a flat one, so it fell back to the same copy `.contiguous()` would have made. Neither call announced which path it took; `data_ptr()` is how you find out after the fact.

## In-place through a view

A view shares storage, and an in-place op writes storage. Put those two facts together: mutating a view mutates whatever else reads that storage, including the tensor the view came from.

**your prediction:**

`row = x[0]` is a view; then `row += 100`. What does `x[0]` print afterward?

In [ ]:
row = x[0]                   # also a view
row += 100                   # in-place: x changes too
print(x[0])                  # tensor([100., 101., 102., 103.])

## Exercise

Write five view chains of your own, mixing `t()`, slicing, `permute`, `expand`, and `reshape`. Predict shape, stride, and `is_contiguous()` for each on paper before you run any of them. The stride oracle at `/gym/pytorch#drills` runs the same drill against real torch at scale; a streak of five clean predictions there is the chapter's bar.

## Mark it run

Read the chapter this lab drills: [kernels.rudrite.com/pytorch/tensors](https://kernels.rudrite.com/pytorch/tensors).